# Build a printable weekly schedule from DonSheet with CRN-based course-number colors

This version keeps the existing printable schedule layout and colored text hierarchy, and adds
course-number color rules based on duplicate CRNs and the values in
`Instruction Method {Inst Method}`:

- **Black**: CRN appears in only one row
- **Green**: same CRN appears in several rows and all relevant rows are `SYNC`
- **Orange**: same CRN appears in several rows and relevant rows include both `SYNC` and `ASYNC`

In [57]:
import re
from pathlib import Path
import pandas as pd
import openpyxl
import xlsxwriter

In [58]:
from pathlib import Path

INPUT_FILE = next(Path("x_SEM_CourseUpdate_LiveFolder_office365").glob("*.xlsx"))
OUTPUT_FILE = Path("Step2_output_Seminary_Smart_Schedule/Weekly_Schedule.xlsx")

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

print("Using input:", INPUT_FILE)
print("Saving output to:", OUTPUT_FILE)

Using input: x_SEM_CourseUpdate_LiveFolder_office365/DonSheet_202641_Fall_v1.0.xlsx
Saving output to: Step2_output_Seminary_Smart_Schedule/Weekly_Schedule.xlsx


In [59]:
def is_valid_time(val):
    return bool(re.fullmatch(r"\d{4}", str(val).strip()))

def fmt_time_ampm(s):
    s = str(s).zfill(4)
    hh, mm = int(s[:2]), int(s[2:])
    ampm = "AM" if hh < 12 else "PM"
    hh12 = hh % 12 or 12
    return f"{hh12}:{mm:02d} {ampm}"

def fmt_range_ampm(start, end):
    return f"{fmt_time_ampm(start)}–{fmt_time_ampm(end)}"

def fmt_time_military(s):
    s = str(s).zfill(4)
    return f"{s[:2]}:{s[2:]}"

def fmt_range_short(start, end):
    return f"{fmt_time_military(start)}-{fmt_time_military(end)}"

def uniq(seq):
    out, seen = [], set()
    for x in seq:
        if pd.isna(x):
            continue
        s = str(x).strip()
        if not s or s in seen:
            continue
        seen.add(s)
        out.append(s)
    return out

def pick_title(row):
    sec = str(row.get("Section Title") or "").strip()
    cat = str(row.get("Catalog Title") or "").strip()
    return sec if sec and sec.lower() not in {"none", cat.lower()} else cat

def room_sort_key(room):
    room = str(room)
    m = re.match(r"([A-Z]+)(\d+)", room)
    return (m.group(1), int(m.group(2))) if m else (room, 0)

def slot_subject(codes):
    m = re.match(r"([A-Z]+)", str(codes or ""))
    return m.group(1) if m else ""

def build_day_pattern(row):
    day_cols = [("MON", "M"), ("TUE", "T"), ("WED", "W"), ("THU", "R"), ("FRI", "F"), ("SAT", "S"), ("SUN", "U")]
    marks = []
    for col, mark in day_cols:
        val = str(row.get(col) or "").strip()
        if val == mark:
            marks.append(mark)
    return "".join(marks)

def format_credit_crn(credits, crns):
    credits = str(credits or "").strip()
    crns = str(crns or "").strip()
    if credits and crns:
        return f"{credits}cr (CRN{crns})"
    if credits:
        return f"{credits}cr"
    if crns:
        return f"(CRN{crns})"
    return ""

def decorate_instructor(name):
    name = str(name or "").strip()
    return f"• {name} •" if name else ""

def normalize_inst_method(val):
    return str(val or "").strip().upper()

BLOCK_ORDER = [
    "Monday Morning",
    "Tuesday Morning",
    "Tuesday 11:30 - 12:30 Chapel",
    "Wednesday Morning",
    "Thursday Morning",
    "Lunch 12:20-2:20 PM / PATH588 Seminary Chorus (CRN 498): 1:30-2:20 PM",
    "Monday Afternoon",
    "Tuesday Afternoon",
    "Wednesday Afternoon",
    "Thursday Afternoon",
    "Supper 5:30 - 6:20 PM",
    "Evening Classes",
]

SPECIAL_ROWS = {
    "Tuesday 11:30 - 12:30 Chapel": "Chapel reserved time",
    "Lunch 12:20-2:20 PM / PATH588 Seminary Chorus (CRN 498): 1:30-2:20 PM": "Lunch window / PATH588 Seminary Chorus if scheduled",
    "Supper 5:30 - 6:20 PM": "Supper reserved time",
}

SUBJECT_FILLS = {
    "CHIS": "#FFF2CC",
    "DSLE": "#E2F0D9",
    "GSEM": "#D9EAF7",
    "MSSN": "#FCE4D6",
    "NTST": "#EADCF8",
    "OTST": "#DDEBF7",
    "PATH": "#E4DFEC",
    "THST": "#F4CCCC",
    "ANEA": "#FFF2F2",
}

def assign_block(day, start, end):
    s = int(start)
    e = int(end)

    if day == "Monday":
        if s < 1220:
            return "Monday Morning"
        elif s < 1800:
            return "Monday Afternoon"
        return "Evening Classes"

    if day == "Tuesday":
        if s < 1130:
            return "Tuesday Morning"
        elif 1130 <= s < 1230 or (s < 1230 and e > 1130):
            return "Tuesday 11:30 - 12:30 Chapel"
        elif s < 1800:
            return "Tuesday Afternoon"
        return "Evening Classes"

    if day == "Wednesday":
        if s < 1220:
            return "Wednesday Morning"
        elif s < 1800:
            return "Wednesday Afternoon"
        return "Evening Classes"

    if day == "Thursday":
        if s < 1220:
            return "Thursday Morning"
        elif s < 1800:
            return "Thursday Afternoon"
        return "Evening Classes"

    return None

def crn_color_rule_for_group(crns, crn_rule_map):
    crn_list = [c.strip() for c in str(crns or "").split(",") if c.strip()]
    group_rules = [crn_rule_map.get(c, "black") for c in crn_list]
    if "orange" in group_rules:
        return "orange"
    if group_rules and all(r == "green" for r in group_rules):
        return "green"
    return "black"

def write_rich_multiline(ws, row, col, rec, fmts, crn_rule_map):
    course_color = crn_color_rule_for_group(rec["CRNs"], crn_rule_map)
    first_fmt = fmts["bold_by_crn_rule"][course_color]

    first_line = rec["Course Codes"]
    second_line = rec["Title"]
    third_line = decorate_instructor(rec["Instructor"])
    fourth_line = format_credit_crn(rec["Credits"], rec["CRNs"])
    fifth_line = rec["DayTime"]

    pieces = [first_fmt, first_line]
    if str(second_line).strip():
        pieces += [fmts["base"], "\n" + str(second_line).strip()]
    if str(third_line).strip():
        pieces += [fmts["instructor"], "\n" + str(third_line).strip()]
    if str(fourth_line).strip():
        pieces += [fmts["credit"], "\n" + str(fourth_line).strip()]
    if str(fifth_line).strip():
        pieces += [fmts["time"], "\n" + str(fifth_line).strip()]
    pieces += [fmts["base"]]
    ws.write_rich_string(row, col, *pieces)

In [60]:
# 1) Read every department sheet into one dataframe
wb = openpyxl.load_workbook(INPUT_FILE, data_only=True)
ignore = {"DropDownMenu", "InstructorMap"}

records = []
for sheet_name in wb.sheetnames:
    if sheet_name in ignore:
        continue

    ws = wb[sheet_name]
    headers = [ws.cell(1, c).value for c in range(1, ws.max_column + 1)]

    for r in range(2, ws.max_row + 1):
        row = {headers[c - 1]: ws.cell(r, c).value for c in range(1, ws.max_column + 1)}
        if row.get("CRN") is None and row.get("Catalog Title") is None:
            continue
        row["_sheet"] = sheet_name
        records.append(row)

df = pd.DataFrame(records)
print(df.shape)

(342, 101)


/opt/anaconda3/lib/python3.13/site-packages/openpyxl/reader/excel.py:237: UserWarning: Data Validation extension is not supported and will be removed
  ws_parser.bind_all()


In [61]:
# 2) Build CRN color rule map from Instruction Method {Inst Method}
inst_col = "Instruction Method {Inst Method}"

crn_method_df = df.copy()
crn_method_df["CRN"] = crn_method_df["CRN"].astype(str).str.strip()
crn_method_df[inst_col] = crn_method_df[inst_col].map(normalize_inst_method)

crn_rule_map = {}

for crn, g in crn_method_df.groupby("CRN", dropna=False):
    crn = str(crn).strip()
    if not crn or crn.lower() == "nan":
        continue

    row_count = len(g)
    methods = set(m for m in g[inst_col].tolist() if m)

    if row_count == 1:
        crn_rule_map[crn] = "black"
    elif methods == {"SYNC"}:
        crn_rule_map[crn] = "green"
    elif "SYNC" in methods and "ASYNC" in methods:
        crn_rule_map[crn] = "orange"
    else:
        crn_rule_map[crn] = "black"

rule_summary = pd.Series(crn_rule_map).value_counts(dropna=False).to_dict()
print("CRN color rule summary:", rule_summary)

CRN color rule summary: {'black': 325, 'green': 2}


In [62]:
# 3) Keep only room-based classes with valid times, then explode to one row per meeting-day
dedupe_cols = [
    "CRN",
    "Subject",
    "Course Number {Crse Num}",
    "Course Section {Seq Crse Num}",
    "Catalog Title",
    "Section Title",
    "Credits {Sect Crs}",
    "Course Beginning Time {Meet Beg Time}",
    "Course Ending Time {Meet End Time}",
    "MON", "TUE", "WED", "THU", "FRI", "SAT", "SUN",
    "Room {Meet Room}",
    "Instructor Name {Instr Name}",
    "Instruction Method {Inst Method}",
]

valid = df[
    df["Course Beginning Time {Meet Beg Time}"].map(is_valid_time)
    & df["Course Ending Time {Meet End Time}"].map(is_valid_time)
    & df["Room {Meet Room}"].notna()
].copy().drop_duplicates(subset=dedupe_cols)

day_map = [
    ("MON", "Monday", "M"),
    ("TUE", "Tuesday", "T"),
    ("WED", "Wednesday", "W"),
    ("THU", "Thursday", "R"),
]

meeting_rows = []
for _, row in valid.iterrows():
    day_pattern = build_day_pattern(row)
    for day_col, day_name, marker in day_map:
        if str(row.get(day_col) or "").strip() == marker:
            section = str(row.get("Course Section {Seq Crse Num}") or "").strip()
            meeting_rows.append(
                {
                    "Day": day_name,
                    "Room": str(row["Room {Meet Room}"]).strip(),
                    "Start": str(row["Course Beginning Time {Meet Beg Time}"]).strip(),
                    "End": str(row["Course Ending Time {Meet End Time}"]).strip(),
                    "CRN": str(row.get("CRN") or "").strip(),
                    "Subject": str(row.get("Subject") or "").strip(),
                    "Course Number": str(row.get("Course Number {Crse Num}") or "").strip(),
                    "Section": section,
                    "Course Code": (
                        f"{str(row.get('Subject') or '').strip()}"
                        f"{str(row.get('Course Number {Crse Num}') or '').strip()}"
                        + (f"-{section}" if section else "")
                    ),
                    "Title": pick_title(row),
                    "Catalog Title": str(row.get("Catalog Title") or "").strip(),
                    "Instructor": str(row.get("Instructor Name {Instr Name}") or "").strip(),
                    "Credits": str(row.get("Credits {Sect Crs}") or "").strip(),
                    "Day Pattern": day_pattern,
                    "DayTime": f"{day_pattern} {fmt_range_short(row['Course Beginning Time {Meet Beg Time}'], row['Course Ending Time {Meet End Time}'])}".strip(),
                    "Inst Method": normalize_inst_method(row.get("Instruction Method {Inst Method}")),
                }
            )

meetings = pd.DataFrame(meeting_rows)
print(meetings.shape)

(98, 16)


In [63]:
# 4) Group cross-listed / stacked classes that share the same room/day/time
grouped_rows = []
for keys, g in meetings.groupby(["Day", "Room", "Start", "End"], sort=False):
    codes = uniq(g["Course Code"])
    titles = uniq(g["Title"])
    catalog_titles = uniq(g["Catalog Title"])
    if len(titles) > 2:
        titles = catalog_titles[:2]
    instrs = uniq(g["Instructor"])
    crns = [c for c in uniq(g["CRN"]) if c not in {"###", "####"}]
    credits = uniq(g["Credits"])
    daytimes = uniq(g["DayTime"])

    grouped_rows.append(
        {
            "Day": keys[0],
            "Room": keys[1],
            "Start": keys[2],
            "End": keys[3],
            "Course Codes": " / ".join(codes),
            "Title": " / ".join(titles),
            "Instructor": " / ".join(instrs),
            "CRNs": ", ".join(crns),
            "Credits": " / ".join(credits),
            "DayTime": " / ".join(daytimes),
            "Time": fmt_range_ampm(keys[2], keys[3]),
        }
    )

slots = (
    pd.DataFrame(grouped_rows)
    .sort_values(["Day", "Start", "Room", "Course Codes"])
    .reset_index(drop=True)
)

slots["Block"] = slots.apply(lambda r: assign_block(r["Day"], r["Start"], r["End"]), axis=1)
slots.head(10)

,Day,Room,Start,End,Course Codes,Title,Instructor,CRNs,Credits,DayTime,Time,Block
0,Monday,N108,0830,1020,CHIS612-001,Intro to Ecclesiastical Latin,NAME NOT FOUND FOR PIDM:,3987,,M 08:30-10:20,8:30 AM–10:20 AM,Monday Morning
1,Monday,N110,0830,1020,OTST552-001,Biblical Hebrew II,"Olariu, Daniel",497,2,M 08:30-10:20,8:30 AM–10:20 AM,Monday Morning
2,Monday,N235,0830,0920,OTST551-001,Biblical Hebrew I,NAME NOT FOUND FOR PIDM:,491,3,MTWR 08:30-09:20,8:30 AM–9:20 AM,Monday Morning
3,Monday,N150,0930,1220,DSLE541-002,Fndtns of Biblicl Spirituality,"Kidder, S Joseph",477,3,M 09:30-12:20,9:30 AM–12:20 PM,Monday Morning
4,Monday,N235,0930,1020,OTST551-002,Biblical Hebrew I,NAME NOT FOUND FOR PIDM:,3119,3,MTWR 09:30-10:20,9:30 AM–10:20 AM,Monday Morning
5,Monday,N235,1030,1220,OTST500-001,Survey of the Old Testament,"Olariu, Daniel",4101,2,M 10:30-12:20,10:30 AM–12:20 PM,Monday Morning
6,Monday,N150,1330,1620,DSLE541-003,Fndtns of Biblicl Spirituality,"Kidder, S Joseph",533,3,M 13:30-16:20,1:30 PM–4:20 PM,Monday Afternoon
7,Monday,S120,1330,1620,DSLE541-001,Fndtns of Biblicl Spirituality,"Ward, Scott R.",3382,3,M 13:30-16:20,1:30 PM–4:20 PM,Monday Afternoon
8,Monday,S115,1700,2000,PATH554-101 / PATH554-102,CPE,NAME NOT FOUND FOR PIDM:,"3098, 3099",,M 17:00-20:00,5:00 PM–8:00 PM,Monday Afternoon
9,Thursday,N108,0830,1020,DSLE678-001,Spiritual Nurture of Children,NAME NOT FOUND FOR PIDM:,4260,,R 08:30-10:20,8:30 AM–10:20 AM,Thursday Morning


In [64]:
# 5) Build the printable workbook with CRN-based course-number colors
rooms_present = sorted(slots["Room"].dropna().unique(), key=room_sort_key)

lower_floor = [r for r in ["N108", "N110", "N150", "S115", "S120"] if r in rooms_present]
main_floor = [r for r in ["N235", "N310"] if r in rooms_present]
upper_floor = [r for r in ["N335", "S340"] if r in rooms_present]
remaining_rooms = [r for r in rooms_present if r not in lower_floor + main_floor + upper_floor]

rooms = lower_floor + main_floor + upper_floor + remaining_rooms
floors = [
    ("LOWER FLOOR", lower_floor),
    ("MAIN FLOOR", main_floor),
    ("UPPER FLOOR", upper_floor),
]
if remaining_rooms:
    floors.append(("OTHER", remaining_rooms))

with pd.ExcelWriter(OUTPUT_FILE, engine="xlsxwriter") as writer:
    book = writer.book
    sheet = book.add_worksheet("Weekly Schedule")
    writer.sheets["Weekly Schedule"] = sheet
    sheet.hide_gridlines(2)

    dark = "#1F4E78"
    thin_gray = "#B7C9D6"
    green = "#2E7D32"
    muted_gray = "#666666"
    orange = "#C55A11"

    title_fmt = book.add_format({
        "bold": True, "font_size": 15, "font_color": "white",
        "align": "center", "valign": "vcenter", "bg_color": dark,
    })
    subtitle_fmt = book.add_format({
        "italic": True, "font_size": 8, "font_color": "#666666",
        "align": "center", "valign": "vcenter", "bg_color": "#F7FAFC"
    })
    floor_fmt = book.add_format({
        "bold": True, "font_size": 10, "align": "center", "valign": "vcenter",
        "bg_color": "#D9EAF7",
        "top": 2, "top_color": dark, "bottom": 1, "bottom_color": thin_gray,
        "left": 1, "left_color": thin_gray, "right": 1, "right_color": thin_gray
    })
    room_header_blank_fmt = book.add_format({
        "bg_color": dark,
        "top": 2, "top_color": dark, "bottom": 2, "bottom_color": dark,
        "left": 1, "left_color": thin_gray, "right": 1, "right_color": thin_gray
    })
    room_header_fmt = book.add_format({
        "bold": True, "font_size": 10, "font_color": "white",
        "align": "center", "valign": "vcenter", "bg_color": dark,
        "top": 2, "top_color": dark, "bottom": 2, "bottom_color": dark,
        "left": 1, "left_color": thin_gray, "right": 1, "right_color": thin_gray
    })
    label_fmt = book.add_format({
        "bold": True, "font_size": 9, "align": "center", "valign": "vcenter",
        "text_wrap": True, "bg_color": "#EAF3FB", "border": 1, "border_color": thin_gray
    })
    special_note_fmt = book.add_format({
        "italic": True, "font_size": 9, "align": "center", "valign": "vcenter",
        "text_wrap": True, "bg_color": "#FFF2CC", "border": 1, "border_color": thin_gray
    })
    blank_fmt = book.add_format({
        "font_size": 8, "align": "left", "valign": "top", "text_wrap": True,
        "bg_color": "white", "border": 1, "border_color": thin_gray
    })

    rich_fmts = {}
    for subj, fill in SUBJECT_FILLS.items():
        common = {
            "font_size": 8, "align": "left", "valign": "top", "text_wrap": True,
            "bg_color": fill, "border": 1, "border_color": thin_gray
        }
        rich_fmts[subj] = {
            "base": book.add_format(common),
            "bold_by_crn_rule": {
                "black": book.add_format({**common, "bold": True, "font_color": "black"}),
                "green": book.add_format({**common, "bold": True, "font_color": green}),
                "orange": book.add_format({**common, "bold": True, "font_color": orange}),
            },
            "instructor": book.add_format({**common, "font_color": dark, "italic": True}),
            "credit": book.add_format({**common, "font_color": green}),
            "time": book.add_format({**common, "font_color": muted_gray}),
        }

    default_common = {
        "font_size": 8, "align": "left", "valign": "top", "text_wrap": True,
        "bg_color": "white", "border": 1, "border_color": thin_gray
    }
    default_fmts = {
        "base": blank_fmt,
        "bold_by_crn_rule": {
            "black": book.add_format({**default_common, "bold": True, "font_color": "black"}),
            "green": book.add_format({**default_common, "bold": True, "font_color": green}),
            "orange": book.add_format({**default_common, "bold": True, "font_color": orange}),
        },
        "instructor": book.add_format({**default_common, "font_color": dark, "italic": True}),
        "credit": book.add_format({**default_common, "font_color": green}),
        "time": book.add_format({**default_common, "font_color": muted_gray}),
    }

    sheet.set_column(0, 0, 24)
    for i in range(1, len(rooms) + 1):
        sheet.set_column(i, i, 18)

    last_col = len(rooms)

    sheet.merge_range(0, 0, 0, last_col, "Printable Weekly Schedule", title_fmt)
    sheet.set_row(0, 24)

    sheet.merge_range(
        1, 0, 1, last_col,
        "Course number colors: black=single-row CRN, green=multi-row SYNC, orange=multi-row mixed SYNC+ASYNC",
        subtitle_fmt
    )
    sheet.set_row(1, 18)

    sheet.write(2, 0, "Time Block", floor_fmt)
    col = 1
    for floor_name, floor_rooms in floors:
        if not floor_rooms:
            continue
        start_col = col
        end_col = col + len(floor_rooms) - 1
        sheet.merge_range(2, start_col, 2, end_col, floor_name, floor_fmt)
        col = end_col + 1
    sheet.set_row(2, 18)

    sheet.write(3, 0, "", room_header_blank_fmt)
    for c, room in enumerate(rooms, start=1):
        sheet.write(3, c, room, room_header_fmt)
    sheet.set_row(3, 18)

    current_row = 4
    for block in BLOCK_ORDER:
        if block in SPECIAL_ROWS:
            sheet.write(current_row, 0, block, label_fmt)
            sheet.merge_range(current_row, 1, current_row, last_col, SPECIAL_ROWS[block], special_note_fmt)
            sheet.set_row(current_row, 22)
            current_row += 1
            continue

        block_df = slots[slots["Block"] == block].copy().sort_values(["Start", "Room", "Course Codes"])
        room_lists = {room: block_df[block_df["Room"] == room].to_dict("records") for room in rooms}
        max_len = max([len(v) for v in room_lists.values()] + [1])

        start_row = current_row
        end_row = current_row + max_len - 1

        if max_len > 1:
            sheet.merge_range(start_row, 0, end_row, 0, block, label_fmt)
        else:
            sheet.write(start_row, 0, block, label_fmt)

        for r in range(start_row, end_row + 1):
            sheet.set_row(r, 64)
            for c, room in enumerate(rooms, start=1):
                recs = room_lists[room]
                slot_index = r - start_row

                if slot_index < len(recs):
                    rec = recs[slot_index]
                    subj = slot_subject(rec["Course Codes"])
                    fmts = rich_fmts.get(subj, default_fmts)
                    write_rich_multiline(sheet, r, c, rec, fmts, crn_rule_map)
                else:
                    sheet.write_blank(r, c, "", blank_fmt)

        current_row = end_row + 1

    sheet.freeze_panes(4, 1)
    sheet.repeat_rows(0, 3)
    sheet.set_landscape()
    sheet.set_paper(1)
    sheet.fit_to_pages(1, 1)
    sheet.set_margins(left=0.2, right=0.2, top=0.3, bottom=0.3)
    sheet.print_area(0, 0, current_row - 1, last_col)

    slots.to_excel(writer, sheet_name="Normalized Meetings", index=False)

    excluded_cols = [
        "CRN",
        "Subject",
        "Course Number {Crse Num}",
        "Course Section {Seq Crse Num}",
        "Catalog Title",
        "Room {Meet Room}",
        "Course Beginning Time {Meet Beg Time}",
        "Course Ending Time {Meet End Time}",
        "Reason",
    ]

    excluded_rows = []
    for _, row in df.iterrows():
        reasons = []
        if pd.isna(row.get("Room {Meet Room}")) or str(row.get("Room {Meet Room}") or "").strip() == "":
            reasons.append("No room")
        if not is_valid_time(row.get("Course Beginning Time {Meet Beg Time}")) or not is_valid_time(row.get("Course Ending Time {Meet End Time}")):
            reasons.append("Invalid or missing time")

        if reasons:
            excluded_rows.append(
                {
                    "CRN": row.get("CRN"),
                    "Subject": row.get("Subject"),
                    "Course Number {Crse Num}": row.get("Course Number {Crse Num}"),
                    "Course Section {Seq Crse Num}": row.get("Course Section {Seq Crse Num}"),
                    "Catalog Title": row.get("Catalog Title"),
                    "Room {Meet Room}": row.get("Room {Meet Room}"),
                    "Course Beginning Time {Meet Beg Time}": row.get("Course Beginning Time {Meet Beg Time}"),
                    "Course Ending Time {Meet End Time}": row.get("Course Ending Time {Meet End Time}"),
                    "Reason": "; ".join(reasons),
                }
            )

    pd.DataFrame(excluded_rows)[excluded_cols].to_excel(writer, sheet_name="Excluded", index=False)

print(f"Saved {OUTPUT_FILE}")

Saved Step2_output_Seminary_Smart_Schedule/Weekly_Schedule.xlsx


/opt/anaconda3/lib/python3.13/site-packages/xlsxwriter/worksheet.py:2353: UserWarning: Can't merge single cell
  warn("Can't merge single cell")
